# Proyecto EVA: Entrenamiento Base CNN (Prueba de Concepto - Versión Completa)
Este notebook define el pipeline completo para la clasificación binaria de patrones de cristalización salival (Helecho vs No Helecho).
Incluye EDA, división estratificada de datos, data augmentation, transfer learning, validación, evaluación estadística e inferencia sobre imágenes nuevas.

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

## 1. Definición del Dataset y EDA

In [ ]:
class EVAFerningDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file).dropna(subset=['Label'])
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        img_name = f"{self.annotations.iloc[index, 0]}.jpg"
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        y_label = torch.tensor(float(self.annotations.iloc[index]['Label']))
        if self.transform:
            image = self.transform(image)
        return image, y_label

def plot_class_distribution(csv_path):
    try:
        df = pd.read_csv(csv_path).dropna(subset=['Label'])
        df['Clase'] = df['Label'].map({1.0: 'Helecho (Positivo)', 0.0: 'No Helecho (Negativo)'})
        plt.figure(figsize=(8, 5))
        sns.countplot(data=df, x='Clase', palette='viridis')
        plt.title('Distribución de Clases')
        plt.show()
    except FileNotFoundError:
        pass

## 2. División Estratificada (Train / Validation) y Transformaciones

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def create_dataloaders(csv_file, root_dir, batch_size=16):
    df = pd.read_csv(csv_file).dropna(subset=['Label'])
    indices = np.arange(len(df))
    labels = df['Label'].values
    
    train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)
    
    train_dataset = EVAFerningDataset(csv_file, root_dir, transform=train_transforms)
    val_dataset = EVAFerningDataset(csv_file, root_dir, transform=val_transforms)
    
    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(val_dataset, val_idx)
    
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader
    
# train_loader, val_loader = create_dataloaders('index_etiquetas.csv', 'Master_Vault')

## 3. Modelo, Función de Pérdida y Optimizador

In [ ]:
def build_model(freeze_base=True):
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    if freeze_base:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, 1)
    return model.to(device)

model = build_model(freeze_base=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

## 4. Ciclo de Entrenamiento y Validación

In [ ]:
def train_and_validate(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    train_losses, val_losses = [], []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        model.eval()
        val_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                predicted = (outputs > 0.0).float()
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        train_loss_avg = running_loss/len(train_loader)
        val_loss_avg = val_loss/len(val_loader)
        val_accuracy = 100 * correct / total
        
        train_losses.append(train_loss_avg)
        val_losses.append(val_loss_avg)
        val_accuracies.append(val_accuracy)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss_avg:.4f} | Val Loss: {val_loss_avg:.4f} | Val Acc: {val_accuracy:.2f}%")
        
    # Graficar Curvas de Entrenamiento
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Pérdida durante Entrenamiento')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(val_accuracies, label='Val Accuracy', color='green')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Precisión en Validación')
    plt.legend()
    plt.show()
    
    return model

# model = train_and_validate(model, train_loader, val_loader, criterion, optimizer, num_epochs=15)

## 5. Evaluación Estadística Final (Matriz de Confusión y Curva ROC)
Evaluamos el modelo entrenado sobre todo el set de validación para extraer métricas clínicas.

In [ ]:
def evaluate_model(model, val_loader):
    model.eval()
    y_true = []
    y_scores = []
    y_pred = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            
            # Aplicar Sigmoid para obtener probabilidades (0 a 1)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            y_scores.extend(probs)
            y_pred.extend(preds)
            y_true.extend(labels.numpy())
            
    # 1. Matriz de Confusión
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['No Helecho', 'Helecho'], 
                yticklabels=['No Helecho', 'Helecho'])
    plt.ylabel('Verdadero')
    plt.xlabel('Predicho')
    plt.title('Matriz de Confusión')
    plt.show()
    
    # 2. Reporte de Clasificación (F1-Score, Recall)
    print("\nReporte de Clasificación Clínica:")
    print(classification_report(y_true, y_pred, target_names=['No Helecho', 'Helecho']))
    
    # 3. Curva ROC
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos')
    plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)')
    plt.title('Curva ROC')
    plt.legend(loc="lower right")
    plt.show()

# evaluate_model(model, val_loader)

## 6. Inferencia en Nueva Fotografía
Carga una imagen externa simulando el uso en producción.

In [ ]:
def predecir_nueva_foto(model, ruta_imagen):
    """
    Toma la ruta de una foto nueva (ej. tomada directo de WhatsApp), 
    aplica las transformaciones base y retorna la probabilidad clínica.
    """
    model.eval()
    try:
        image = Image.open(ruta_imagen).convert("RGB")
    except FileNotFoundError:
        print(f"Error: No se encontró la imagen en {ruta_imagen}")
        return
        
    # Aplicar estrictamente val_transforms (sin augmentation)
    input_tensor = val_transforms(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        probabilidad = torch.sigmoid(output).item()
        
    diagnostico = "Patrón Fértil (Helecho)" if probabilidad > 0.5 else "No Helecho"
    
    # Mostrar la imagen con su resultado
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(f"Diagnóstico: {diagnostico}\nConfianza: {probabilidad*100:.2f}%")
    plt.axis('off')
    plt.show()

# Ruta de ejemplo para probar:
# predecir_nueva_foto(model, "/ruta/a/tu/nueva_foto_whatsapp.jpg")